# Intelligent Early-Stage Diabetes Risk Stratification for Rural Health Posts in Nepal: A Comparative Evaluation of Machine Learning Models

## Import Dataset and Generate EDA

In [ ]:
# import libraries
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
import numpy as np
pd.set_option('display.max_columns', None)

In [ ]:
# load dataset
df = pd.read_csv("./datasets/final.csv")
df.head()

### Count of targets

In [ ]:
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x='diabetes_target', palette="Set2")
plt.title('Target Class Distribution (0: Healthy, 1: Diabetic/Prediabetic)', fontsize=12, fontweight='bold')
plt.xlabel('Diabetes Status', fontsize=10)
plt.ylabel('Patient Count', fontsize=10)
plt.xticks([0, 1], ['Non-Diabetic (0)', 'Diabetic / Prediabetic (1)'])

# annotate counts on top of bars
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=10)

plt.tight_layout()
plt.savefig("./resources/target_class_distribution.png")
plt.show()
plt.close()

### Co-relataion heatmap

In [ ]:
plt.figure(figsize=(14, 10))
# computes correlation matrix of df
corr = df.corr()

sns.heatmap(
    corr,
    cmap='coolwarm', 
    annot=True, 
    fmt=".2f", 
    linewidths=0.5, 
    )
plt.title('Correlation Heatmap of Health Indicators', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("./resources/correlation_heatmap_of_health_indicators.png")
plt.show()
plt.close()

### BMI Distribution BoxPlot

In [ ]:
plt.figure(figsize=(8, 5))

sns.kdeplot(
    data=df[df["diabetes_target"] == 0],
    x="bmi",
    fill=True,
    alpha=0.5,
    label="Non-Diabetic"
)

sns.kdeplot(
    data=df[df["diabetes_target"] == 1],
    x="bmi",
    fill=True,
    alpha=0.5,
    label="Diabetic / Prediabetic"
)

plt.title("BMI Density by Diabetes Status", fontsize=12, fontweight="bold")
plt.xlabel("Body Mass Index (BMI)")
plt.ylabel("Density")
plt.legend()

plt.tight_layout()
plt.savefig("./resources/bmi_density_plot.png")
plt.show()
plt.close()

### Split and Scale
Scaling is required, so the value comes between same scale range. Split for making train, val and test sets.

In [ ]:
# import libraries
from sklearn.preprocessing import StandardScaler

In [ ]:
# Split data to train, test, val
train_len = int(len(df) * 0.65)
val_len = train_len + int(len(df) * 0.2)

train = df.iloc[:train_len]
val = df.iloc[train_len:val_len]
test = df.iloc[val_len:]

print(f"Train Set shape = {train.shape}")
print(f"Validation Set shape = {val.shape}")
print(f"Test Set shape = {test.shape}")


In [ ]:
# continuous value
continuous = [
    "bmi",
    "poor_mental_health_days",
    "poor_physical_health_days",
    "general_health_rating",
    "age_category",
    "education_level",
    "income_level"
]
# scaler 
scaler = {}
# continuous looping
for col in continuous:
    scaler[col] = StandardScaler()
    train[col] = scaler[col].fit_transform(train[[col]]).flatten()
    val[col] = scaler[col].transform(val[[col]]).flatten()
    test[col] = scaler[col].transform(test[[col]]).flatten()


## Predict Diabetic or Not Using Decision Tree and Evaluate

In [ ]:
# import library
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_curve, roc_auc_score, confusion_matrix

In [ ]:
# use decision tree 
dt = DecisionTreeClassifier(
    criterion = "gini",
)

In [ ]:
dt.fit(train.drop(columns=["diabetes_target"]), train["diabetes_target"])

In [ ]:
X_train = train.drop(columns=["diabetes_target"])
y_train = train["diabetes_target"]

X_val = val.drop(columns=["diabetes_target"])
y_val = val["diabetes_target"]

X_test = test.drop(columns=["diabetes_target"])
y_test = test["diabetes_target"]

### Train Decision Trees and Store Metrics

In [ ]:
dt_metrics = []

for depth in range(1, 16):
    dt = DecisionTreeClassifier(
        max_depth=depth,
        random_state=41,
        criterion="gini"
    )

    dt.fit(X_train, y_train)

    train_pred = dt.predict(X_train)
    val_pred = dt.predict(X_val)

    metric = {
        "depth":depth,
        # train 
        "train_accuracy":accuracy_score(y_train, train_pred),
        "train_precision":precision_score(y_train, train_pred),
        "train_recall":recall_score(y_train, train_pred),
        "train_f1":f1_score(y_train, train_pred),
        # val
        "val_accuracy":accuracy_score(y_val, val_pred),
        "val_precision":precision_score(y_val, val_pred),
        "val_recall":recall_score(y_val, val_pred),
        "val_f1":f1_score(y_val, val_pred),
    }
    # append metrics
    dt_metrics.append(metric)
    
dt_metrics = pd.DataFrame(dt_metrics)

### Plot comparative line charts for train vs validation for each metics (DECISION TREE)

In [ ]:
# figure size
plt.figure(figsize=(12, 10))

# subplots for each
for i, col in enumerate(["accuracy", "precision", "recall", "f1"]):
    # create a subplot
    plt.subplot(2, 2, i+1)
    plt.title(f"{col.capitalize()} Score Comparison for Every Depth")
    sns.lineplot(dt_metrics, x=f"depth", y=f"train_{col}")
    sns.lineplot(dt_metrics, x=f"depth", y=f"val_{col}")
    plt.yticks([j/100 for j in range(60, 92, 2)])
    plt.xticks([j for j in range(16)])
    plt.xlabel("depth")
    plt.ylabel(col)
    
plt.tight_layout()
# save
plt.savefig("./resources/decision_tree_metric_plots.png")
plt.show()
plt.close()

### Evaluation as per chart
In case of each of the measures, the performance of training (blue line) improves gradually with an increase in the depth of trees up to reaching its peak value of about 0.85-0.87 at depth 15 while, the performance of validation/testing (orange line) improves gradually up to depth 7-8 after which it begins to deteriorate significantly, showing an obvious indication of overfitting at higher depths. On the whole, the maximum possible performance is achieved at depth 7-8 where the performance measures of validation become the highest as Accuracy reaches its peak at about 0.74 (at depth 7), Precision reaches its peak at about 0.73 (depth 7-8), Recall is stabilized at about 0.76-0.77 (depth 7-9) and F1 score is optimized at validation at about 0.75 (at depth 7). At and after depth 8, there occurs a distinct variation between training and validation performance measures with recall dropping to about 0.71 and accuracy to about 0.69 at depth 15 on validation, thereby suggesting that depth 7 or 8 is the optimum for the model.

## Find the best min_sample_leafs

In [ ]:
dt_metrics_msl = [] # metrics of min sample leaves
decision_trees = []

for leaf in range(1, 51, 5):
    dt = DecisionTreeClassifier(
        max_depth=8,
        random_state=41,
        criterion = "gini",
        min_samples_leaf=leaf
        
    )

    dt.fit(X_train, y_train)

    train_pred = dt.predict(X_train)
    val_pred = dt.predict(X_val)

    metric = {
        "min_samples_leaf":leaf,
        # train 
        "train_accuracy":accuracy_score(y_train, train_pred),
        "train_precision":precision_score(y_train, train_pred),
        "train_recall":recall_score(y_train, train_pred),
        "train_f1":f1_score(y_train, train_pred),
        # val
        "val_accuracy":accuracy_score(y_val, val_pred),
        "val_precision":precision_score(y_val, val_pred),
        "val_recall":recall_score(y_val, val_pred),
        "val_f1":f1_score(y_val, val_pred),
    }
    # append metrics
    dt_metrics_msl.append(metric)
    decision_trees.append({"min_samples_leaf":leaf, "model":dt})
    
dt_metrics_msl = pd.DataFrame(dt_metrics_msl)

### Plot comparative charts for min_sample_leaf in Decision Tree

In [ ]:
# figure size
plt.figure(figsize=(12, 10))

# subplots for each
for i, col in enumerate(["accuracy", "precision", "recall", "f1"]):
    # create a subplot
    plt.subplot(2, 2, i+1)
    plt.title(f"{col.capitalize()} Score Comparison for Every Depth")
    sns.lineplot(dt_metrics_msl, x=f"min_samples_leaf", y=f"train_{col}")
    sns.lineplot(dt_metrics_msl, x=f"min_samples_leaf", y=f"val_{col}")
    plt.yticks([j/100 for j in range(60, 92, 2)])
    plt.xticks([j for j in range(0, 51, 5)])
    plt.xlabel("min_samples_leaf")
    plt.ylabel(col)
    
plt.tight_layout()
# save
plt.savefig("./resources/decision_tree_metric_plots_features_leaf.png")
plt.show()
plt.close()

## Predict Diabetic or Not Using Random Forest and Evaluate

In [ ]:
# import library
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# all forest metrics
forest_metrics = []

for i in range(1, 16):

    rf = RandomForestClassifier(
        n_estimators=10,
        max_depth=i,
        random_state=41
    )

    rf.fit(X_train, y_train)

    train_pred = rf.predict(X_train)
    val_pred = rf.predict(X_val)
    
    metric = {
        "depth":i,
        # train 
        "train_accuracy":accuracy_score(y_train, train_pred),
        "train_precision":precision_score(y_train, train_pred),
        "train_recall":recall_score(y_train, train_pred),
        "train_f1":f1_score(y_train, train_pred),
        # val
        "val_accuracy":accuracy_score(y_val, val_pred),
        "val_precision":precision_score(y_val, val_pred),
        "val_recall":recall_score(y_val, val_pred),
        "val_f1":f1_score(y_val, val_pred),
    }
    forest_metrics.append(metric)
    
forest_metrics = pd.DataFrame(forest_metrics)

### Plot comparative line charts for train vs validation for each metrics (RANDOM FOREST)

In [ ]:
# figure size
plt.figure(figsize=(12, 10))

# subplots for each
for i, col in enumerate(["accuracy", "precision", "recall", "f1"]):
    # create a subplot
    plt.subplot(2, 2, i+1)
    plt.title(f"{col.capitalize()} Score Comparison for Every Depths")
    sns.lineplot(forest_metrics, x=f"depth", y=f"train_{col}")
    sns.lineplot(forest_metrics, x=f"depth", y=f"val_{col}")
    plt.xticks([j for j in range(0,20,2)])
    plt.xlabel("depth")
    plt.ylabel(col)
    
plt.tight_layout()
# save
plt.savefig("./resources/random_forest_metric_on_different_depths.png")
plt.show()
plt.close()

### Evaluation
Max_depth=8 is selected as the optimal value for the hyperparameters tuning of Random Forest model from the empirical validation graphs. From the overall analysis of all the metrics, shallow trees (1-3 depths) are under-fitting with poor performance on all the metrics (accuracy ~0.726, F1 ~0.724), whereas trees with greater than 10 depths have higher generalization gaps between training and validation metric values, which indicates over-fitting though validation F1 score slightly improves with depth 11 (~0.759). Max_depth=8 provides the optimal balance of variance/bias among all values of max_depth. It provides excellent results with all the metrics (accuracy ~0.743, precision ~0.726, recall ~0.780, F1 ~0.752) while having the lowest generalization gap (~0.012). This is made possible by keeping max_depth=8 so that decision tree is able to learn nonlinearity but not noise.

### Find best estimators for selected max_depth

In [ ]:
# all forest metrics with depth 8 as it was it gave the highest scores without overfitting previously
forest_metrics_est = []
rfs = []
for i in range(1, 100):

    rf = RandomForestClassifier(
        n_estimators=i,
        max_depth=8,
        random_state=41
    )

    rf.fit(X_train, y_train)

    train_pred = rf.predict(X_train)
    val_pred = rf.predict(X_val)
    
    metric = {
        "estimators":i,
        # train 
        "train_accuracy":accuracy_score(y_train, train_pred),
        "train_precision":precision_score(y_train, train_pred),
        "train_recall":recall_score(y_train, train_pred),
        "train_f1":f1_score(y_train, train_pred),
        # val
        "val_accuracy":accuracy_score(y_val, val_pred),
        "val_precision":precision_score(y_val, val_pred),
        "val_recall":recall_score(y_val, val_pred),
        "val_f1":f1_score(y_val, val_pred),
    }
    forest_metrics_est.append(metric)
    rfs.append(rf)
    
forest_metrics_est = pd.DataFrame(forest_metrics_est)

In [ ]:
plt.figure(figsize=(12, 10))

for i, col in enumerate(["accuracy", "precision", "recall", "f1"]):
    plt.subplot(2, 2, i+1)

    plt.title(f"{col.capitalize()} Score Comparison for Every Estimators")

    sns.lineplot(
        data=forest_metrics_est,
        x="estimators",
        y=f"train_{col}",
        label="Train"
    )

    sns.lineplot(
        data=forest_metrics_est,
        x="estimators",
        y=f"val_{col}",
        label="Validation"
    )

    plt.xticks(range(10, 101, 10))
    plt.yticks([j/100 for j in range(66, 81,2)])
    plt.xlabel("Estimators")
    plt.ylabel(col.capitalize())

    plt.legend()

plt.tight_layout()

plt.savefig("./resources/random_forest_metric_on_different_estimators.png")
plt.show()
plt.close()

### Evaluation
From the empirical validation curves plotted for different numbers of estimators, the optimal value for this hyperparameter, which would be used finally in implementation, is n_estimators = 40 (in the range 30-40). For each of the four performance measures, a small number of estimators (1-10) produces high variance and poor performance, with validation accuracy being approximately 0.722 and F1 score 0.732. The more the number of trees in the forest, the better and more stable becomes the validation performance, reaching about 30-40 estimators, when validation accuracy gets approximately 0.748, precision becomes 0.729, recall 0.792, and F1 score 0.759. Beyond 40 up to 100 estimators, there are no further improvements in performance; validation accuracy improves marginally to around 0.750 and F1 score to 0.761, while training and validation curves stay exactly parallel to each other with a small generalization gap (0.010).

## Model Testing using Test Set on Random Forest

Based on the empirical performance assessment of the model, the selected principal model for the evaluation on the test set is the Random Forest classifier (n_estimators = 40, max_depth = 8). Even though the alternative model is the single Decision Tree at the depth of 7, the ensemble model performs much better in terms of all significant parameters on the validation set, improving Recall to ~0.792 and increasing F1 score to ~0.759. From the perspective of the architecture, using 40 decorrelated decision trees helps to minimize the variance and increase resistance to the sample noise as well as reduce the generalization gap to ~0.010. Therefore, using the Random Forest on the new holdout set will provide the most robust performance of the model and its generalization ability.

In [ ]:
# extract exact model
random_forest = rfs[39]

In [ ]:
# run predictions
y_pred = random_forest.predict(X_test)
y_prob = random_forest.predict_proba(X_test)[:,1]

### Calculate and show confusion matrix

In [ ]:
# confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

In [ ]:
# percentage
cm_percent = cm / cm.sum() * 100

# combine count + percentage
labels = [
    [f"{cm[i,j]}\n({cm_percent[i,j]:.2f}%)" for j in range(cm.shape[1])]
    for i in range(cm.shape[0])
]

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=labels,
    fmt="",
    cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.savefig(
    "./resources/confusion_matrix_count_percentage_random_forest.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Evaluation
The confusion matrix reveals very good classification results with respect to the test set ($N = 10,605$), with the optimized Random Forest classifier being able to predict 7,960 items in order to obtain a very high accuracy equal to 75.06%. In detail, the classifier was able to detect 4,216 true positive instances (39.75% of total number of test items) and 3,744 true negative instances (35.30%). Therefore, it achieved a very high sensitivity (recall) of 79.2% for the positive class and positive predictive value (precision) of 73.3%. False positives were detected in the same proportion as expected at random, i.e., 1,537 items (14.49%) were classified as positives even though they were negative items, while 1,108 items (10.45%) were classified as negatives but were positive ones. The low rate of false negatives compared to false positives means that the classifier has a higher sensitivity toward positive items.

### Calculate and show AUC

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
# AUC score
auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", auc)

In [ ]:
plt.figure(figsize=(7,5))

sns.lineplot(
    x=fpr,
    y=tpr,
    label=f"ROC-AUC = {auc:.3f}"
)

# random classifier baseline
sns.lineplot(
    x=[0, 1],
    y=[0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.savefig(
    "./resources/roc_auc_curve_random_forest.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Evaluation
In the ROC analysis for the optimized Random Forest classifier (n_estimators = 40, max_depth = 8), there is a significant discriminative capacity on the holdout test data, where the Area Under the Curve (ROC-AUC) is 0.827. The shape of the curve has a steep increase in the initial stages and achieves a True Positive Rate (Sensitivity) of 0.60 at a False Positive Rate of 0.15, which means the model can detect the majority of the positive cases at the same time maintaining a lower number of false alarms. Additionally, the large gap from the baseline (diagonal line) with text AUC = 0.50 shows good separation capability of the classifier through all decision thresholds in terms of probability.

In [ ]:
test_metrics_rf = {
    # Confusion matrix
    "True Positives":tp,
    "False Positives":fp,
    "False Negatives":fn,
    "True Negatives":tn,
    # metrics
    "Accuracy Score": accuracy_score(y_test, y_pred),
    "Precision Score": precision_score(y_test, y_pred),
    "Recall Score": recall_score(y_test, y_pred),
    "F1 Score": f1_score(y_test, y_pred),
    # auc
    "Roc-Auc":auc
}

# show all values
test_metrics_rf

### Viewing where error happened

In [ ]:
test_results = X_test.copy()

test_results["actual"] = y_test.values
test_results["predicted"] = y_pred

# False negatives: Actual diabetes but predicted non-diabetes
false_negatives = test_results[
    (test_results["actual"] == 1) & 
    (test_results["predicted"] == 0)
]

# False positives: Actual non-diabetes but predicted diabetes
false_positives = test_results[
    (test_results["actual"] == 0) & 
    (test_results["predicted"] == 1)
]

print("False Negatives:")
display(false_negatives.head())

print("False Positives:")
display(false_positives.head())

### Random Forest overall analysis and Conclusion
The final evaluation of the optimized Random Forest classifier (n_estimators=40, max_depth=8) using the hold-out test set is another clear confirmation of high generalization ability and balanced performance of the model. The evaluation of the confusion matrix with 4,216 True Positives, 3,744 True Negatives, 1,537 False Positives, and 1,108 False Negatives leads to the conclusion about 75.06% (0.7506) accuracy of the classification in the test set. The model demonstrates 73.28% (0.7328) Precision and 79.19% (0.7919) Recall scores; thus, its F1 Score as a harmonic mean equals 76.12% (0.7612). In other words, the model is able to identify positive cases and avoid making false negative predictions. The discrimination capacity of the model under consideration at different decision thresholds is proven by ROC-AUC score of 0.8268 (0.827).

## Model Evaluation for Decision Tree
In order to establish the baseline model, a standalone Decision Tree was applied at the optimum depth of the tree (max_depth = 8). At this depth, the model guards against the issue of underfitting due to insufficiently small depths (less than 4), while avoiding the variance problem after passing the critical tree depth threshold (greater than 8). At this depth level, the numerical performance of the model is very high for all of the metrics considered, and includes the following metrics: Validation Accuracy = 0.738, Precision = 0.730, Recall = 0.768, and F1 Score = 0.748. The similarity between training and validation results at this depth level is an indication of good generalization ability. The Decision Tree at depth level 8 will be the baseline model for comparison with other models, such as Random Forest.

In [ ]:
# extract exact model
decision_tree = next(item["model"] for item in decision_trees if item["min_samples_leaf"] == 21)

In [ ]:
# run predictions
y_pred_dt = decision_tree.predict(X_test)
y_prob_dt = decision_tree.predict_proba(X_test)[:,1]

### Calculate and Show Confusion Matrix

In [ ]:
# confusion matrix
cm = confusion_matrix(y_test, y_pred_dt)
tn, fp, fn, tp = cm.ravel()

In [ ]:
# percentage
cm_percent = cm / cm.sum() * 100

# combine count + percentage
labels = [
    [f"{cm[i,j]}\n({cm_percent[i,j]:.2f}%)" for j in range(cm.shape[1])]
    for i in range(cm.shape[0])
]

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=labels,
    fmt="",
    cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.savefig(
    "./resources/confusion_matrix_count_percentage_decision_tree.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Evaluation
The confusion matrix for the isolated Decision Tree baseline, where max_depth=8, shows good performance on test set (N = 10,605) by making 7,869 correct predictions to reach the test accuracy of 74.20%. The model detected 4,078 true positive samples (38.45% of all test samples) and 3,791 true negative samples (35.75%), providing sensitivity (recall) equal to 76.60% and precision equal to 73.24% (F1 score is 74.88%). The misclassifications consisted of 1,490 false positive (14.05%) and 1,246 false negative (11.75%). Comparing with the tuned Random Forest model (75.06% accuracy, 79.19% recall), one can see that Decision Tree has larger false-negative rate and lower recall in general, proving that, even though single tree is a strong baseline, an ensemble manages to recover missing positive samples.

### Calculate and Show AUC

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob_dt)
# AUC score
auc = roc_auc_score(y_test, y_prob_dt)

print("ROC-AUC:", auc)

In [ ]:
plt.figure(figsize=(7,5))

sns.lineplot(
    x=fpr,
    y=tpr,
    label=f"ROC-AUC = {auc:.3f}"
)

# random classifier baseline
sns.lineplot(
    x=[0, 1],
    y=[0, 1],
    linestyle="--",
    label="Random Classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.savefig(
    "./resources/roc_auc_curve_decision_tree.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

### Evaluation
In the current scenario, the area under ROC curve for the base Decision Tree classifier (max_depth = 8) on the hold-out test data equals 0.818. In addition, the plot demonstrates steady increase in ROC curve with the True Positive Rate (Sensitivity) reaching around 0.60, while the False Positive Rate is about 0.16. Although, this model demonstrates good class separation in terms of ROC-AUC compared to random guessing (text{AUC} = 0.50), it still lags behind the optimized Random Forest classifier (text{AUC} = 0.827). The lower ROC-AUC value demonstrates that the one-dimensional nature of decision-making in one tree is inferior to the smoothed approach employed by ensemble. Overall, AUC of 0.818 indicates that this model is quite a good baseline model.

In [ ]:
test_metrics_dt = {
    # Confusion matrix
    "True Positives":tp,
    "False Positives":fp,
    "False Negatives":fn,
    "True Negatives":tn,
    # metrics
    "Accuracy Score": accuracy_score(y_test, y_pred_dt),
    "Precision Score": precision_score(y_test, y_pred_dt),
    "Recall Score": recall_score(y_test, y_pred_dt),
    "F1 Score": f1_score(y_test, y_pred_dt),
    # auc
    "Roc-Auc":auc
}

# show all values
test_metrics_dt

### Decision Tree overall analysis and conclusion
The results of the evaluation of the baseline Decision Tree classifier (max_depth = 8) based on the held-out dataset indicate that there is a good baseline level of performance. Based on the results from the confusion matrix with 4,078 True Positives, 3,791 True Negatives, 1,490 False Positives, and 1,246 False Negatives, the accuracy of the classifier is calculated to be 74.20% (0.7420). In addition to that, it produces Precision of 73.24% (0.7324), Recall of 76.60% (0.7660), and F1 Score of 74.88% (0.7488). Its capability for discriminating predictions at different probability thresholds is measured by ROC-AUC value of 0.8182 (0.818). When comparing those metrics to those of the optimized Random Forest (75.06% accuracy, 79.19% recall, 0.8268 ROC-AUC), it can be seen that although Decision Tree gives a stable baseline

### Random Forest's Feature's Weight

In [ ]:
# Get feature importance from Random Forest
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": random_forest.feature_importances_
})

# Sort features by importance
feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

# Display importance values
print(feature_importance)

# Plot feature importance
plt.figure(figsize=(10, 8))

sns.barplot(
    data=feature_importance,
    x="Importance",
    y="Feature"
)

plt.xlabel("Feature Importance Score")
plt.ylabel("Health Indicators")
plt.title("Feature Importance Distribution of Random Forest")

plt.tight_layout()

plt.savefig(
    "./resources/random_forest_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()